# 36 — FIM Topology Universality

**UNPRECEDENTED:** Does FIM work on ANY network topology?

We tested FIM only on K_nm (exponential decay). But if FIM is a universal
mechanism, it should work on:
1. **Random (Erdős-Rényi)** — no structure
2. **Small-world (Watts-Strogatz)** — high clustering + short paths
3. **Scale-free (Barabási-Albert)** — hubs + long tail
4. **Ring** — minimal connectivity
5. **Complete** — maximal connectivity
6. **Star** — one hub

If FIM synchronises ALL of these, it's truly universal.
If only some, topology tells us WHERE self-observation matters.

In [ ]:
import json
from pathlib import Path

import numpy as np

N = 32
rng_topo = np.random.default_rng(42)


def fim_gradient_all(phases):
    n = len(phases)
    z = np.exp(1j * phases)
    mu = np.angle(np.mean(z))
    R = np.abs(np.mean(z))
    phase_diff = (mu - phases + np.pi) % (2 * np.pi) - np.pi
    sensitivity = min(1.0 / (1.0 - R**2 + 0.01), 50.0)
    return (1.0 / n) * np.sin(phase_diff) * sensitivity


def make_ring(N):
    K = np.zeros((N, N))
    for i in range(N):
        K[i, (i + 1) % N] = K[(i + 1) % N, i] = 1.0
    return K


def make_complete(N):
    return np.ones((N, N)) - np.eye(N)


def make_star(N):
    K = np.zeros((N, N))
    for i in range(1, N):
        K[0, i] = K[i, 0] = 1.0
    return K


def make_erdos_renyi(N, p=0.3, seed=42):
    rng = np.random.default_rng(seed)
    K = np.zeros((N, N))
    for i in range(N):
        for j in range(i + 1, N):
            if rng.random() < p:
                K[i, j] = K[j, i] = 1.0
    return K


def make_small_world(N, k=4, p_rewire=0.3, seed=42):
    rng = np.random.default_rng(seed)
    K = np.zeros((N, N))
    # Start with ring lattice
    for i in range(N):
        for j in range(1, k // 2 + 1):
            K[i, (i + j) % N] = K[(i + j) % N, i] = 1.0
    # Rewire
    for i in range(N):
        for j in range(1, k // 2 + 1):
            if rng.random() < p_rewire:
                new_j = rng.integers(0, N)
                while new_j == i or K[i, new_j] > 0:
                    new_j = rng.integers(0, N)
                K[i, (i + j) % N] = K[(i + j) % N, i] = 0
                K[i, new_j] = K[new_j, i] = 1.0
    return K


def make_scale_free(N, m=2, seed=42):
    rng = np.random.default_rng(seed)
    K = np.zeros((N, N))
    # Start with m+1 fully connected
    for i in range(m + 1):
        for j in range(i + 1, m + 1):
            K[i, j] = K[j, i] = 1.0
    degrees = np.sum(K, axis=1)
    # Add nodes one at a time
    for new_node in range(m + 1, N):
        probs = degrees[:new_node] / (degrees[:new_node].sum() + 1e-10)
        targets = rng.choice(new_node, size=min(m, new_node), replace=False, p=probs)
        for t in targets:
            K[new_node, t] = K[t, new_node] = 1.0
        degrees = np.sum(K, axis=1)
    return K


topologies = {
    "Ring": make_ring(N),
    "Complete": make_complete(N),
    "Star": make_star(N),
    "Erdos-Renyi": make_erdos_renyi(N, p=0.3),
    "Small-World": make_small_world(N, k=6, p_rewire=0.3),
    "Scale-Free": make_scale_free(N, m=2),
}

omega = rng_topo.standard_normal(N) * 0.5

for name, K in topologies.items():
    edges = int(np.sum(K > 0) / 2)
    density = edges / (N * (N - 1) / 2)
    print(f"{name:>15}: {edges} edges, density={density:.3f}")

In [ ]:
def simulate_R_topo(K, omega, K_scale, fim_lambda, N, dt=0.02, T=100.0, noise=0.05, seed=42):
    K_eff = K * K_scale
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, N)
    n_steps = int(T / dt)
    R_tail = []
    for s in range(n_steps):
        diff = theta[None, :] - theta[:, None]
        coupling = np.sum(K_eff * np.sin(diff), axis=1) / N
        dphi = omega + coupling
        if fim_lambda > 0:
            dphi += fim_lambda * fim_gradient_all(theta)
        theta = (theta + dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)) % (2 * np.pi)
        if s >= n_steps * 3 // 4:
            R_tail.append(float(np.abs(np.mean(np.exp(1j * theta)))))
    return float(np.mean(R_tail))


# Test: K_scale=1 (normalised), λ ∈ {0, 1, 5}
print(f"{'Topology':>15} {'R(λ=0)':>8} {'R(λ=1)':>8} {'R(λ=5)':>8} {'FIM boost':>10}")
universality_data = []

for name, K in topologies.items():
    R_vals = {}
    for lam in [0, 1, 5]:
        Rs = [simulate_R_topo(K, omega, 1.0, lam, N, seed=s) for s in [42, 123, 456]]
        R_vals[lam] = float(np.mean(Rs))

    boost = R_vals[5] - R_vals[0]
    universality_data.append(
        {
            "topology": name,
            **{f"R_lam{l}": round(v, 4) for l, v in R_vals.items()},
            "boost": round(boost, 4),
        }
    )
    print(f"{name:>15} {R_vals[0]:8.4f} {R_vals[1]:8.4f} {R_vals[5]:8.4f} {boost:+10.4f}")

# Verdict
all_boost = [d["boost"] for d in universality_data]
min_boost = min(all_boost)
print(f"\nMin FIM boost across topologies: {min_boost:+.4f}")
if min_boost > 0.1:
    print("FIM is TOPOLOGY-UNIVERSAL: improves sync on every tested network.")
else:
    weak = [d["topology"] for d in universality_data if d["boost"] < 0.1]
    print(f"FIM is NOT universal. Weak on: {', '.join(weak)}")

In [ ]:
# Save
output = {
    "experiment": "FIM topology universality — does self-observation work on any network?",
    "N": N,
    "data": universality_data,
}
out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/topology_universality_2026-03-29.json"
)
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"Results saved to {out_path}")